In [23]:
import pandas as pd
import numpy as np
import pandas_ta as ta

# CSVファイルを読み込む
df = pd.read_csv(
    'data/AAPL.csv',
    parse_dates=['Date'],
    date_format='%m/%d/%Y',
    index_col='Date'
)

# データを日付順にソート
df = df.sort_index()

In [24]:
# 必要な列を選択
ohlcv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
# 両方のリターンを計算
# df_pct = df[ohlcv_cols].pct_change()
df_log = np.log(df[ohlcv_cols] / df[ohlcv_cols].shift(1))
df_log * 10000

,Open,High,Low,Close,Volume
Date,,,,,
2015-02-26,NaN,NaN,NaN,NaN,NaN
2015-02-27,93.916562,-22.949826,127.920128,-151.025682,-3867.194543
2015-03-02,-57.859371,-22.235010,4.677633,48.922637,-2539.367310
2015-03-03,-22.462346,-58.506710,-16.381298,20.893798,-2411.069558
2015-03-04,10.850191,3.087849,17.940023,-63.590753,-1781.903295
...,...,...,...,...,...
2025-02-19,20.867011,33.795508,54.449574,16.348555,-4160.935675
2025-02-20,11.437910,31.250660,46.347363,39.127827,34.931228
2025-02-21,41.149806,77.098894,37.997227,-11.396476,4984.194643


In [42]:
df_log_open = np.log(df[['Open', 'Volume']] / df[['Open', 'Volume']].shift(1)) * 10000
# 一度に計算する方法
cols = ['High', 'Low', 'Close']
# 対数リターンの場合
df_log_from_open = pd.DataFrame({
        f"{col}_From_Open": np.log(df[col] / df['Open']) * 10000
        for col in cols
    },
    index=df.index
)
df_ofs = pd.concat([df_log_open, df_log_from_open], axis=1)
df_ofs[1:]


,Open,Volume,High_From_Open,Low_From_Open,Close_From_Open
Date,,,,,
2015-02-27,93.916562,-3867.194543,43.750310,-136.309422,-119.168786
2015-03-02,-57.859371,-2539.367310,79.374670,-73.772418,-12.386779
2015-03-03,-22.462346,-2411.069558,43.330306,-67.691369,30.969365
2015-03-04,10.850191,-1781.903295,35.567965,-60.601537,-43.471579
2015-03-05,-40.360191,5795.220395,13.212608,-221.759500,-170.206862
...,...,...,...,...,...
2025-02-19,20.867011,-4160.935675,55.026939,-61.481837,8.579659
2025-02-20,11.437910,34.931228,74.839689,-26.572384,36.269575
2025-02-21,41.149806,4984.194643,110.788777,-29.724964,-16.276708


In [30]:
# カスタム戦略を作成
my_strategy = ta.Strategy(
    name="基本指標",
    ta=[
        # trend
        {"kind": "sma", "length": 5},
        {"kind": "sma", "length": 25},
        {"kind": "sma", "length": 75},
        # momentum
        {"kind": "rsi"},
        {"kind": "roc"},
        # volatility
        {"kind": "atr"},
        {"kind": "bbands"},
        # volume
        {"kind": "obv"},
        {"kind": "vwap"},
    ]
)

# 戦略を適用
df.ta.strategy(my_strategy)
df[75:]

,Close,Volume,Open,High,Low,SMA_5,SMA_25,SMA_75,RSI_14,ROC_10,...,BBM_5_2.0,BBU_5_2.0,BBB_5_2.0,BBP_5_2.0,ATRr_14,OBV,VWAP_D,DCL_20_20,DCM_20_20,DCU_20_20
Date,,,,,,,,,,,,,,,,,,,,,
2015-06-15,31.730,175587840,31.5250,31.8100,31.4275,31.9490,32.292944,31.878565,42.861661,-2.769223,...,31.9490,32.343089,2.466987,0.222144,0.479244,-1.623975e+09,31.655833,31.405,32.32375,33.2425
2015-06-16,31.900,125774600,31.7575,31.9625,31.5925,31.9580,32.305744,31.875699,45.793777,-1.815943,...,31.9580,32.345084,2.422454,0.425081,0.471413,-1.498200e+09,31.818333,31.405,32.32375,33.2425
2015-06-17,31.825,131435600,31.9300,31.9700,31.6850,31.8790,32.320096,31.869732,44.703855,-2.167230,...,31.8790,32.169010,1.819444,0.406900,0.458053,-1.629636e+09,31.826667,31.405,32.32375,33.2425
2015-06-18,31.970,141455960,31.8075,32.0775,31.8050,31.8435,32.338796,31.864799,47.314636,-1.144094,...,31.8435,32.010880,1.051265,0.877883,0.444758,-1.488180e+09,31.950833,31.405,32.32375,33.2425
2015-06-19,31.650,217446120,31.9275,31.9550,31.6000,31.8150,32.315296,31.858332,42.540994,-1.593471,...,31.8150,32.044434,1.442301,0.140420,0.439403,-1.705626e+09,31.735000,31.405,32.32375,33.2425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-19,244.870,32204220,244.6600,246.0100,243.1604,242.4680,233.204400,236.886400,60.668817,5.184708,...,242.4680,248.569834,5.033105,0.696826,5.675701,2.396930e+09,244.680133,219.790,233.49000,247.1900
2025-02-20,245.830,32316910,244.9400,246.7800,244.2900,244.2600,233.706400,237.096133,61.640571,5.746978,...,244.2600,247.151311,2.367405,0.771503,5.448151,2.429247e+09,245.633333,221.410,234.30000,247.1900
2025-02-21,245.550,53197430,245.9500,248.6900,245.2200,245.0640,234.013600,237.358000,61.165892,5.286854,...,245.0640,246.133101,0.872507,0.727294,5.306854,2.376050e+09,246.486667,221.410,235.05000,248.6900
